# SRQ-FLY Priority 4: 10-task versus 20-task robustness
Five paired CIFAR-100 train-only replicates. This notebook never materializes `test.pt`; it changes only task grouping while fixing validation samples, class order, projection, WTA codes, Ridge, and implementation.

In [ ]:
# Edit path/source values only. Do not edit protocol, seeds, schedules, or gates.
REPO_GIT_URL='https://github.com/ZaPhat206/SOHO-CL.git'
REPO_BRANCH='experiment/soho-selfcontained'
WORK_DIR='/content/SOHO-CL'
FEATURE_CACHE_DIR='/content/srq_priority4_cifar_features'
WTA_CACHE_ROOT='/content/srq_priority4_wta'
OUTPUT_DIR='/content/srq_priority4_task_frequency'
BATCH_SIZE=128
NUM_WORKERS=2

In [ ]:
# Fresh clone, dependencies, GPU check, and immutable source verification.
import hashlib,json,os,shutil,subprocess,sys,time
from pathlib import Path
os.chdir('/content')
repo=Path(WORK_DIR)
if repo.exists(): shutil.rmtree(repo)
subprocess.run(['git','clone','--branch',REPO_BRANCH,'--single-branch',REPO_GIT_URL,WORK_DIR],check=True)
os.chdir(WORK_DIR)
subprocess.run([sys.executable,'-m','pip','install','-q','-r','requirements-kaggle.txt','kagglehub','huggingface_hub','pandas','matplotlib'],check=True)
import torch
assert torch.cuda.is_available(),'Select Runtime -> Change runtime type -> T4 GPU.'
def sha(path): return hashlib.sha256(Path(path).read_bytes()).hexdigest()
CONFIG='configs/srq_fly_priority4_cifar100_task_frequency_train_only.json'
RUNNER='tools/srq_fly_priority4_task_frequency.py'
EXPECTED={
 CONFIG:'d4165de428a71cebd3d1272ca623b0604b092e876fdcd159fd599aac00d0bd6c',
 RUNNER:'463d1dcab13b2f37a9e014b5b171d239f75b37548835d7a19022b062992daf39',
 'methods/srq_fly_optimized/learner.py':'40edac2e2cc88faac549f5c87217f3143d815bf53ecad8a37dfdb22c112691ae',
 'methods/srq_fly_optimized/storage.py':'9d288a3661985da657371e8581f406825d4a8d5e6e0c63381aacda8484490986',
 'tools/srq_fly_d0.py':'60566c92512a97ae49d981b746f6c91c477b3d8dade3194ec6fa4b51f9c3619a',
 'tools/twa_fly_pilot.py':'ee1efe6f793ea7ec6ba5ba4489dafefa76a8c2342447a3076bb3fae0316f088a'}
for path,expected in EXPECTED.items(): assert sha(path)==expected,(path,sha(path),expected)
assert not subprocess.check_output(['git','status','--porcelain'],text=True).strip(),'Repository must start clean.'
print('GPU:',torch.cuda.get_device_name(0))
print('commit:',subprocess.check_output(['git','rev-parse','HEAD'],text=True).strip())
print('SOURCE HASH GATE: PASS')

In [ ]:
# Protocol, pairing, learner, storage, and historical-regression gates.
command=[sys.executable,'-m','pytest','-q','tests/test_srq_fly_priority4_task_frequency.py','tests/test_srq_fly_priority3_direct_control.py','tests/test_srq_fly_priority2d_equivalence.py']
completed=subprocess.run(command)
assert completed.returncode==0,'Correctness gate failed; return complete pytest output.'
print('SRQ-FLY PRIORITY-4 CORRECTNESS GATE: PASS')

In [ ]:
# Download the exact frozen checkpoint and processed CIFAR-100 source.
import kagglehub
from huggingface_hub import hf_hub_download
CHECKPOINT_PATH=hf_hub_download(repo_id='timm/vit_base_patch16_224.augreg2_in21k_ft_in1k',filename='model.safetensors')
assert Path(CHECKPOINT_PATH).stat().st_size==346284714
assert sha(CHECKPOINT_PATH)=='32aa17d6e17b43500f531d5f6dc9bc93e56ed8841b8a75682e1bb295d722405b'
CIFAR_ROOT=kagglehub.dataset_download('zaphat206/cifar-100')
print('checkpoint:',CHECKPOINT_PATH)
print('CIFAR-100:',CIFAR_ROOT)

In [ ]:
# Reuse or extract TRAIN features only. Held-out test features are forbidden.
cache=Path(FEATURE_CACHE_DIR)
if not (cache/'train.pt').is_file():
    command=[sys.executable,'-u','tools/experiment_runner.py','--extract-features-only','--extract-train-only','--root',CIFAR_ROOT,'--backbone-checkpoint',CHECKPOINT_PATH,'--backbone-checkpoint-size','346284714','--backbone-checkpoint-sha256','32aa17d6e17b43500f531d5f6dc9bc93e56ed8841b8a75682e1bb295d722405b','--feature-cache-dir',FEATURE_CACHE_DIR,'--output-dir','/content/unused_priority4','--dataset','CIFAR-100','--model-name','vit_base_patch16_224','--data-augmentation','vit','--seed','2025','--num-classes','100','--num-tasks','20','--device','cuda','--batch-size',str(BATCH_SIZE),'--num-workers',str(NUM_WORKERS)]
    print('TRAIN FEATURE EXTRACTION START',flush=True)
    subprocess.run(command,check=True)
assert (cache/'train.pt').is_file() and not (cache/'test.pt').exists()
metadata=json.loads((cache/'metadata.json').read_text())
assert metadata['feature_dim']==768 and metadata['finite'] is True
print('TRAIN CACHE READY:',metadata.get('train_shape'),'| test.pt absent')

In [ ]:
# Five replicates x two schedules. Safe to rerun: completed source-matched units resume.
command=[sys.executable,'-u',RUNNER,'run','--config',CONFIG,'--feature-cache-dir',FEATURE_CACHE_DIR,'--code-cache-root',WTA_CACHE_ROOT,'--output-dir',OUTPUT_DIR,'--device','cuda']
print('PRIORITY-4 START: 5 paired replicates; each runs 10-task and 20-task schedules.',flush=True)
completed=subprocess.run(command)
result_path=Path(OUTPUT_DIR)/'priority4_results.json'
assert completed.returncode==0 and result_path.is_file(),'Priority-4 failed; return complete output without editing config.'
summary=json.loads(result_path.read_text())
assert summary['uses_test_set'] is False
print('DECISION:',summary['status'])
print(json.dumps(summary['gates'],indent=2))
print('Added-frequency loss:',json.dumps(summary['summaries']['added_frequency_loss_pp'],indent=2))

In [ ]:
# Paired tables and task-frequency plots; all values are train-validation only.
import pandas as pd,matplotlib.pyplot as plt,numpy as np
rows=[]
for item in summary['replicate_comparisons']:
    rows.append({'replicate':item['replicate']['id'],'Exact aligned AIA T10':item['aligned_exact_aia_10'],'SRQ aligned AIA T10':item['aligned_srq_aia_10'],'Exact aligned AIA T20':item['aligned_exact_aia_20'],'SRQ aligned AIA T20':item['aligned_srq_aia_20'],'added frequency loss pp':item['added_frequency_loss_pp'],'Exact schedule agreement':item['exact_final_prediction_schedule_agreement']})
table=pd.DataFrame(rows); display(table)
means=[table['Exact aligned AIA T10'].mean(),table['SRQ aligned AIA T10'].mean(),table['Exact aligned AIA T20'].mean(),table['SRQ aligned AIA T20'].mean()]
fig,axes=plt.subplots(1,2,figsize=(12,4.5))
axes[0].bar(['Exact T10','SRQ T10','Exact T20','SRQ T20'],means); axes[0].set_ylabel('Aligned validation AIA (%)'); axes[0].tick_params(axis='x',rotation=25)
axes[1].bar(table.replicate.astype(str),table['added frequency loss pp']); axes[1].axhline(.25,color='red',ls='--',label='locked gate'); axes[1].axhline(0,color='black',lw=.8); axes[1].set(xlabel='Replicate',ylabel='Added SRQ loss at T20 (pp)'); axes[1].legend()
fig.tight_layout(); FIGURE=Path('/content/srq_fly_priority4_summary.png'); fig.savefig(FIGURE,dpi=180,bbox_inches='tight'); plt.show()
units=[json.loads((Path(OUTPUT_DIR)/name).read_text()) for name in summary['unit_files']]
fig,axis=plt.subplots(figsize=(7,4.5))
for schedule in (10,20):
    chosen=[row for row in units if row['num_tasks']==schedule]
    curves=np.array([[d['srq_minus_exact_pp'] for d in row['task_diagnostics']] for row in chosen])
    seen=np.array([d['seen_classes'] for d in chosen[0]['task_diagnostics']])
    axis.plot(seen,curves.mean(0),marker='o',label=f'{schedule} tasks')
axis.axhline(0,color='black',lw=.8); axis.set(xlabel='Seen classes',ylabel='SRQ - Exact accuracy (pp)',title='Effect of update frequency'); axis.grid(alpha=.25); axis.legend()
fig.tight_layout(); CURVE_FIGURE=Path('/content/srq_fly_priority4_curves.png'); fig.savefig(CURVE_FIGURE,dpi=180,bbox_inches='tight'); plt.show()

In [ ]:
# Export evidence only; feature and WTA sample caches are deliberately excluded.
bundle=Path('/content/srq_fly_priority4_task_frequency_train_only')
if bundle.exists(): shutil.rmtree(bundle)
bundle.mkdir()
shutil.copytree(OUTPUT_DIR,bundle/'results')
shutil.copy2(CONFIG,bundle/'locked_config.json')
shutil.copy2('docs/research/SRQ_FLY_PRIORITY4_TASK_FREQUENCY_PROTOCOL.md',bundle/'protocol.md')
shutil.copy2(FIGURE,bundle/FIGURE.name); shutil.copy2(CURVE_FIGURE,bundle/CURVE_FIGURE.name)
manifest={'artifact':'srq_fly_priority4_task_frequency_train_only','uses_test_set':False,'git_commit':subprocess.check_output(['git','rev-parse','HEAD'],text=True).strip(),'source_hashes':EXPECTED,'summary_sha256':sha(Path(OUTPUT_DIR)/'priority4_results.json')}
(bundle/'artifact_manifest.json').write_text(json.dumps(manifest,indent=2))
archive=shutil.make_archive(str(bundle),'zip',root_dir=bundle)
print('ARTIFACT:',archive,'sha256=',sha(archive))
from google.colab import files
files.download(archive)